# Baseline scenario: Linear Regression

`scenario_id = baseline_linear_regression`

Plain OLS (on standardized features) per (target, horizon) cell, on the
baseline feature set — seasonality, lags, and (load only) weekday
features. No weather or capacity information yet. See
`docs/modeling_plan.md` for the full plan and `dhf/features.py` for the
feature definitions.

This notebook and `05_baseline_lightgbm.ipynb` follow an identical
structure (features used -> MAE/RMSE -> actual-vs-predicted -> error
analysis -> feature importance) so the two models are easy to compare
section by section; the shared computation/plotting code lives in
`dhf/reporting.py`.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import pandas as pd

from dhf.paths import ProjPaths
from dhf.reporting import (
    describe_feature_sets,
    fit_and_evaluate,
    plot_actual_vs_predicted,
    plot_importance_heatmaps,
    plot_mae_by_horizon,
    plot_rolling_error,
)
from dhf.scenarios import save_scenario_metrics
from dhf.splits import EXAMPLE_WINDOWS
from dhf.viz import KNOWN_SCENARIO_COLORS, MODEL_LABELS

paths = ProjPaths()
paths.ensure_directories()

SCENARIO_ID = "baseline_linear_regression"
COLOR = KNOWN_SCENARIO_COLORS[SCENARIO_ID]
MODEL_LABEL = MODEL_LABELS["linear_regression"]

panel = pd.read_parquet(paths.target_panel_file)

## Features used

Feature columns are identical across all three horizons for a given
target (only their *values* change — see `dhf/features.py`), so it's
enough to list them once per target.

In [2]:
for target, features in describe_feature_sets(panel).items():
    print(f"{target:15s} ({len(features)} features): {', '.join(features)}")

pv              (8 features): hour_sin, hour_cos, doy_sin, doy_cos, lag_1h, lag_2h, lag_24h, lag_48h
wind_onshore    (8 features): hour_sin, hour_cos, doy_sin, doy_cos, lag_1h, lag_2h, lag_24h, lag_48h
wind_offshore   (8 features): hour_sin, hour_cos, doy_sin, doy_cos, lag_1h, lag_2h, lag_24h, lag_48h
load            (15 features): hour_sin, hour_cos, doy_sin, doy_cos, lag_1h, lag_2h, lag_24h, lag_48h, lag_weekday_match, weekday_Tuesday, weekday_Wednesday, weekday_Thursday, weekday_Friday, weekday_Saturday, weekday_Sunday


## Fit and evaluate

One `LinearRegression` (on standardized features, via a
`StandardScaler` pipeline — needed so coefficient magnitude is a
meaningful importance signal later, without changing the predictions
themselves) per (target, horizon) cell, trained on data before the
validation cutoff and scored on the validation period only (test is
reserved — see `docs/modeling_plan.md`).

In [3]:
def make_model():
    return make_pipeline(StandardScaler(), LinearRegression())


def importance_fn(model, feature_names):
    coefs = model.named_steps["linearregression"].coef_
    return pd.Series(np.abs(coefs), index=feature_names)


results, val_predictions, importances = fit_and_evaluate(panel, make_model, importance_fn)

save_scenario_metrics(
    paths,
    scenario_id=SCENARIO_ID,
    title=MODEL_LABEL,
    description=(
        "Linear regression (standardized features) on seasonality (daily/annual) + "
        "short lags (1h, 2h, 24h, 48h) + load-only weekday features. No weather/capacity data."
    ),
    notebook="04_baseline_linear_regression",
    valid=True,
    results=results,
)
print(f"Saved -> {paths.scenario_results_file(SCENARIO_ID)}")

Saved -> /home/chris/research/delu-headline-forecast/data/processed/results/baseline_linear_regression.parquet


## Results overview

MAE is the primary metric; RMSE is reported alongside for context.

In [4]:
overview = results.set_index(["target", "horizon_days"])[["mae", "rmse", "n_train", "n_val"]].round(1)
overview

mae    rmse  n_train  n_val
target        horizon_days                                
pv            1             1950.9  3448.2    41982  17520
              7             2612.3  4693.0    41982  17520
              14            2708.1  4553.6    41982  17520
wind_onshore  1             6352.7  8175.5    41982  17520
              7             7351.8  9381.3    41982  17520
              14            7347.5  9380.2    41982  17520
wind_offshore 1             1458.5  1742.6    41982  17520
              7             1520.8  1789.5    41982  17520
              14            1525.7  1794.9    41982  17520
load          1             2155.3  3071.5    41886  17520
              7             2522.3  3750.5    41862  17520
              14            2941.1  4297.8    41862  17520

## Validation MAE by horizon

```{figure} ../../output/images/04_lr_mae_by_horizon.png
:name: fig-04-lr-mae-by-horizon
Linear regression validation MAE by forecast horizon, one panel per target.
```

In [5]:
fig = plot_mae_by_horizon(results, color=COLOR)
fig.suptitle(f"{MODEL_LABEL} — validation MAE by forecast horizon", y=1.04)
fig.savefig(paths.images_path / "04_lr_mae_by_horizon.png", dpi=150, bbox_inches="tight")

## Actual vs. predicted — summer window

```{figure} ../../output/images/04_lr_actual_vs_predicted_summer.png
:name: fig-04-lr-actual-vs-predicted-summer
Actual (black) vs. linear regression (blue), 2024-07-01 .. 2024-07-21.
```

In [6]:
fig = plot_actual_vs_predicted(val_predictions, EXAMPLE_WINDOWS["summer"], color=COLOR, model_label=MODEL_LABEL)
fig.suptitle(f"{MODEL_LABEL} — actual vs. predicted, summer window", y=1.04)
fig.savefig(paths.images_path / "04_lr_actual_vs_predicted_summer.png", dpi=150, bbox_inches="tight")

## Actual vs. predicted — winter window

```{figure} ../../output/images/04_lr_actual_vs_predicted_winter.png
:name: fig-04-lr-actual-vs-predicted-winter
Actual (black) vs. linear regression (blue), 2025-01-01 .. 2025-01-21.
```

In [7]:
fig = plot_actual_vs_predicted(val_predictions, EXAMPLE_WINDOWS["winter"], color=COLOR, model_label=MODEL_LABEL)
fig.suptitle(f"{MODEL_LABEL} — actual vs. predicted, winter window", y=1.04)
fig.savefig(paths.images_path / "04_lr_actual_vs_predicted_winter.png", dpi=150, bbox_inches="tight")

## Error analysis — full validation period

7-day rolling mean absolute error over the full ~2-year validation
period, one line per horizon. Reveals whether error is roughly constant
over time (homoskedastic) or clusters seasonally / grows with some
regime — e.g. PV error should track the summer/winter irradiance swing.

```{figure} ../../output/images/04_lr_rolling_error.png
:name: fig-04-lr-rolling-error
7-day rolling mean absolute error over the full validation period, by
target and horizon.
```

In [8]:
fig = plot_rolling_error(val_predictions)
fig.suptitle(f"{MODEL_LABEL} — 7-day rolling validation MAE", y=1.06)
fig.savefig(paths.images_path / "04_lr_rolling_error.png", dpi=150, bbox_inches="tight")

## Feature importance

Absolute standardized coefficient, normalized to a share of the total
per (target, horizon) fit, so columns are directly comparable.

```{figure} ../../output/images/04_lr_feature_importance.png
:name: fig-04-lr-feature-importance
Feature importance share (|standardized coefficient|) by target and
horizon.
```

In [9]:
fig = plot_importance_heatmaps(importances)
fig.suptitle(f"{MODEL_LABEL} — feature importance", y=1.02)
fig.savefig(paths.images_path / "04_lr_feature_importance.png", dpi=150, bbox_inches="tight")